In [12]:
import json
import sqlite3
from pathlib import Path
import pandas as pd 


jlas = 'jlas-development-capital'
cadence = 'cadence-equity-partners'
scci = 'scci'
json_company_name = scci
json_file_name = f"company_data_{json_company_name}-limited.json"

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
DB_PATH = BASE_DIR / "data" / "database" / "cf.db"





with open(DATA_DIR / json_file_name) as f:
    data = json.load(f)
company_name = data["company_profile"]["company_name"]
profile = data['company_profile']

officer_networks = data['officer_networks']
officers = data['officers']
'''
for items in officers['items']:
    print(items['officer_role'])
    
    
    print(officer_id)
    
    '''


accounts_filings = [
    filing
    for filing in data["filing_history"]["items"]
    if filing["type"] == "AA"
]

latest_accounts = accounts_filings[0]

print(latest_accounts["date"])
print(latest_accounts["description_values"]["made_up_date"])

document_metadata = latest_accounts["links"]["document_metadata"]
print(document_metadata)

2026-03-16
2025-09-30
https://document-api.company-information.service.gov.uk/document/9N7sJvwmt0UtuWyTEEZ4q3BRBLLktpAuRGa4ZCxRSxI


In [14]:

with sqlite3.connect(DB_PATH, timeout=10) as conn:
    cursor = conn.cursor()



#create company table that stores 'company name' and 'company number'

cursor.execute("""
CREATE TABLE IF NOT EXISTS company (
    company_name TEXT,
    company_number TEXT UNIQUE
    
)

""")




#create officers table that stores 'appointment_id', 'officer name', 'appointed_on', 'resigned on'

#cursor.execute('''DROP TABLE officers''')

#table creation officers

cursor.execute("""
CREATE TABLE IF NOT EXISTS officers (
    appointment_id TEXT PRIMARY KEY,
    officer_id TEXT,
    name TEXT NOT NULL,
    company_name TEXT,
    role TEXT,
    appointed_on TEXT,
    resigned_on TEXT,
    person_number TEXT
    
    )
""")

#data insertion in officers table

for items in officers['items']:
    
    officer_link = items['links']['officer']['appointments']
    officer_id = officer_link.split("/")[-2]

    appointment_link = items['links']['self']
    appointment_id = appointment_link.split("/")[-1]

    cursor.execute('''
        INSERT INTO officers (
        appointment_id,
        officer_id,
        name, 
        company_name, 
        role, 
        appointed_on, 
        person_number
    )
    VALUES(?,?,?,?,?,?,?)
    ON CONFLICT(appointment_id) DO NOTHING
    
''', (
    appointment_id,
    officer_id,
    items['name'],
    company_name,
    items['officer_role'],
    items['appointed_on'],
    items['person_number']
    
))

            

cursor.execute('''
    INSERT OR IGNORE INTO officers (company_name)
    values(?)
''', (
    profile['company_name'],
))

if cursor.rowcount == 0:
    print(f"Duplicate appointment skipped: {appointment_id}")
else:
    print(f"Appointment inserted: {appointment_id}")

        
conn.commit()


with sqlite3.connect(DB_PATH) as conn:
    df_company = pd.read_sql_query("SELECT * FROM company", conn)
    df_officers = pd.read_sql_query("SELECT * FROM officers", conn)

df_officers



Duplicate appointment skipped: 9dvGltvlyu82vd4jlY6BlbQMR8s


,appointment_id,officer_id,name,company_name,role,appointed_on,resigned_on,person_number
0,mHtKCk84r9QM1BrdHJiM0q7x3fQ,9rTELS9WKtzj_obTqKwoza2cPuo,"MCCOMIE, Linda Maria",SCCI LIMITED,secretary,2004-06-10,None,46683410001
1,bBAbEcW9BZsA6yYjYPvCDOllt2s,slvP-8J_k11EqsbEZ0EmkjZnFCw,"HUMPHRIES, Andrew Stephen, Dr",SCCI LIMITED,director,2004-06-10,None,59739140001
2,EZzZyk88sdn5npLyy_vNHTS4Am4,9rTELS9WKtzj_obTqKwoza2cPuo,"MCCOMIE, Linda Maria",SCCI LIMITED,director,2010-06-11,None,46683410001
3,25PXLssWvOxPb9SPA_B7FMcUMsQ,8d_bnTiwfxh8JIr3YfuwkmkWkCg,SWIFT INCORPORATIONS LIMITED,SCCI LIMITED,corporate-nominee-secretary,2004-06-10,None,900008300001
4,5eow7PFj-lJ9vTB6-RQGMf6NEok,pOjCel8bGPQmL2KJVSlip7WdtsY,"HUMPHRIES, Patricia Jane",SCCI LIMITED,director,2004-06-10,None,98258080001
5,eOsc8tBIpJ-E1VQB7zjuFOoK7i4,F8HJ2k25BM9kHbAvajZmkjJpCPU,"WILDING, Janice Caroline",SCCI LIMITED,director,2004-06-10,None,98258070001
6,ZCzANFNHt4iWjevQTQ7twUoUomI,hvkkJJsGj9I4BBPEEjVn_gY7qew,"WILDING, Richard David",SCCI LIMITED,director,2004-06-10,None,98258060001
7,CepUZqsrtAI0BJ7J29SBqsp25gQ,c5JRw_DLtq8LwgXblyCB5dxlw-o,"HOOFT, Pieter Johannes",CADENCE EQUITY PARTNERS LIMITED,director,2016-06-17,None,148426590001
8,kTM8bydwMw1NdWv6rYOCLTw_bKQ,7-1PyKHAxwApBPHt48vc2H1YcJI,"SZPIRO, James Lucien Alexander",CADENCE EQUITY PARTNERS LIMITED,director,2016-06-17,None,136504450001
9,ZY9PZwVMvZztljIOWlmXg9DZs6U,G1anRrYkdjwSscEDPGU1BTQ2gaU,"SZPIRO, James Lucien Alexander",JLAS DEVELOPMENT CAPITAL LIMITED,secretary,2013-09-12,None,181245630001


In [16]:
conn.close()

In [ ]:
cursor.execute("""
INSERT OR IGNORE INTO company (company_name, company_number) 
VALUES (?, ?)
   
""", (
    profile["company_name"],
    profile["company_number"],
   
))



In [ ]:

cursor.execute('''
    INSERT OR IGNORE INTO officers (appointment_id)
    VALUES(?)
    
''', (
    officer_id,
    
))


In [59]:
print(json.dumps(officer_networks["officer_name"], indent=2))

[
  {
    "officer_name": "MCCOMIE, Linda Maria",
    "companies": [
      {
        "name": "MCCOMIE CONSULTANCY SERVICES LIMITED",
        "number": "04560866",
        "role": "director",
        "appointed": "2002-10-11",
        "resigned": null
      },
      {
        "name": "MCCOMIE CONSULTANCY SERVICES LIMITED",
        "number": "04560866",
        "role": "secretary",
        "appointed": "2002-10-11",
        "resigned": null
      },
      {
        "name": "ALINA ORCHESTRA LIMITED",
        "number": "10926978",
        "role": "director",
        "appointed": "2017-08-22",
        "resigned": "2025-04-03"
      },
      {
        "name": "MKCO LIMITED",
        "number": "01268436",
        "role": "director",
        "appointed": "2008-07-29",
        "resigned": "2012-05-17"
      }
    ]
  },
  {
    "officer_name": "HUMPHRIES, Andrew Stephen, Dr",
    "companies": [
      {
        "name": "INTEGRATED SKY LIMITED",
        "number": "08317019",
        "role": "dire

In [20]:
for key, value in data.items():
    print(f"{key}: {type(value).__name__}")

company_profile: dict
search_results: dict
officers: dict
psc_ownership: dict
filing_history: dict
charges: dict
insolvency: dict
officer_networks: list


In [5]:
cursor.execute("PRAGMA table_info(officers)")
cursor.fetchall()

[(0, 'appointment_id', '', 0, None, 1),
 (1, 'company_number', 'TEXT', 1, None, 0),
 (2, 'officer_name', 'TEXT', 0, None, 0),
 (3, 'officer_role', 'TEXT', 0, None, 0),
 (4, 'appointed_on', 'TEXT', 0, None, 0),
 (5, 'resigned_on', 'TEXT', 0, None, 0)]

401
